In [5]:
import pandas as pd

xls = pd.ExcelFile("./data/netflix_user_behavior_dataset.xlsx")
print(xls.sheet_names)

dataset = pd.read_excel(xls, "dataset")
dataset.to_csv("./data/netflix.csv", index=False)


['netflix_user_behavior_dataset', 'dataset', 'gender', 'country', 'subscription_type', 'payment_method', 'primary_device', 'favorite_genre', 'churned']


In [6]:
df = pd.read_csv("./data/netflix.csv")

df = df.drop(columns=["user_id"])
df.head()

,age,gender,country,account_age_months,subscription_type,monthly_fee,payment_method,primary_device,devices_used,favorite_genre,avg_watch_time_minutes,watch_sessions_per_week,binge_watch_sessions,completion_rate,rating_given,content_interactions,recommendation_click_rate,days_since_last_login,churned
0,56,1,1,17,1,15.99,1,1,1,1,220,17,3,60,1.7,5,66,16,0
1,46,2,2,20,1,12.99,1,2,2,2,76,15,4,71,4.6,7,78,14,0
2,32,3,2,25,2,15.99,1,2,2,3,215,6,13,33,2.0,27,29,41,0
3,60,1,3,37,1,12.99,1,3,3,4,280,4,9,58,1.2,9,23,22,0
4,25,1,4,23,3,12.99,1,4,3,5,261,15,9,64,1.3,49,56,54,0


In [7]:
df.isnull().sum()


age                          0
gender                       0
country                      0
account_age_months           0
subscription_type            0
monthly_fee                  0
payment_method               0
primary_device               0
devices_used                 0
favorite_genre               0
avg_watch_time_minutes       0
watch_sessions_per_week      0
binge_watch_sessions         0
completion_rate              0
rating_given                 0
content_interactions         0
recommendation_click_rate    0
days_since_last_login        0
churned                      0
dtype: int64

In [9]:
df["engagement"] = df["watch_sessions_per_week"] * df["avg_watch_time_minutes"]
df["is_active"] = df["days_since_last_login"] < 15
df["binge_ratio"] = df["binge_watch_sessions"] / df["watch_sessions_per_week"]
df["interaction_score"] = df["content_interactions"] * df["recommendation_click_rate"]
df["account_age_years"] = df["account_age_months"] / 12
df.head()

,age,gender,country,account_age_months,subscription_type,monthly_fee,payment_method,primary_device,devices_used,favorite_genre,...,rating_given,content_interactions,recommendation_click_rate,days_since_last_login,churned,engagement,is_active,binge_ratio,interaction_score,account_age_years
0,56,1,1,17,1,15.99,1,1,1,1,...,1.7,5,66,16,0,3740,False,0.176471,330,1.416667
1,46,2,2,20,1,12.99,1,2,2,2,...,4.6,7,78,14,0,1140,True,0.266667,546,1.666667
2,32,3,2,25,2,15.99,1,2,2,3,...,2.0,27,29,41,0,1290,False,2.166667,783,2.083333
3,60,1,3,37,1,12.99,1,3,3,4,...,1.2,9,23,22,0,1120,False,2.250000,207,3.083333
4,25,1,4,23,3,12.99,1,4,3,5,...,1.3,49,56,54,0,3915,False,0.600000,2744,1.916667


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop("churned", axis=1)
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

df["prediction"] = model.predict(X)
df.head(20)

,age,gender,country,account_age_months,subscription_type,monthly_fee,payment_method,primary_device,devices_used,favorite_genre,...,content_interactions,recommendation_click_rate,days_since_last_login,churned,engagement,is_active,binge_ratio,interaction_score,account_age_years,prediction
0,56,1,1,17,1,15.99,1,1,1,1,...,5,66,16,0,3740,False,0.176471,330,1.416667,0
1,46,2,2,20,1,12.99,1,2,2,2,...,7,78,14,0,1140,True,0.266667,546,1.666667,0
2,32,3,2,25,2,15.99,1,2,2,3,...,27,29,41,0,1290,False,2.166667,783,2.083333,0
3,60,1,3,37,1,12.99,1,3,3,4,...,9,23,22,0,1120,False,2.250000,207,3.083333,0
4,25,1,4,23,3,12.99,1,4,3,5,...,49,56,54,0,3915,False,0.600000,2744,1.916667,0
5,38,2,5,23,1,12.99,1,2,2,2,...,15,36,18,0,1012,False,0.090909,540,1.916667,0
6,56,3,4,11,2,15.99,2,2,1,2,...,45,46,2,0,999,True,1.333333,2070,0.916667,0
7,36,3,6,46,1,15.99,1,4,3,1,...,42,35,59,0,2930,False,0.200000,1470,3.833333,0
8,40,2,6,20,2,12.99,2,1,2,6,...,15,24,31,0,1414,False,1.285714,360,1.666667,0
9,28,2,7,7,3,15.99,3,4,2,4,...,11,17,31,1,492,False,2.500000,187,0.583333,1


In [13]:
df["churned"].value_counts(normalize=True)


churned
0    0.80072
1    0.19928
Name: proportion, dtype: float64

In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [15]:
df["low_activity"] = df["watch_sessions_per_week"] < 5
df["inactive_user"] = df["days_since_last_login"] > 30

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop("churned", axis=1)
y = df["churned"]

# encoding
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [19]:
preds = model.predict(X_test)

import pandas as pd
pd.Series(preds).value_counts()

0    8024
1    1976
Name: count, dtype: int64

In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8007
           1       1.00      0.99      1.00      1993

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [22]:
df.corr()


,age,gender,country,account_age_months,subscription_type,monthly_fee,payment_method,primary_device,devices_used,favorite_genre,...,days_since_last_login,churned,engagement,is_active,binge_ratio,interaction_score,account_age_years,prediction,low_activity,inactive_user
age,1.000000,-0.007493,-0.000960,-0.003321,-0.000881,-0.001249,0.005220,-0.001841,0.003628,0.005392,...,0.002316,-0.002417,-0.002494,-0.001231,-0.005157,-0.004984,-0.003321,-0.002584,0.003721,0.000551
gender,-0.007493,1.000000,-0.000573,0.001923,0.000293,0.003103,0.003509,0.000253,-0.004353,-0.009862,...,-0.006031,0.007211,-0.004933,0.006928,0.004416,0.004562,0.001923,0.006964,0.005008,-0.008895
country,-0.000960,-0.000573,1.000000,-0.000390,-0.006897,-0.001772,-0.006522,0.000674,0.002021,-0.009971,...,0.001808,-0.001041,0.001959,-0.002107,0.003636,-0.001650,-0.000390,-0.001027,0.001529,0.001970
account_age_months,-0.003321,0.001923,-0.000390,1.000000,-0.000036,0.002198,0.000750,0.000688,-0.002959,0.007346,...,0.002224,-0.006852,-0.013809,-0.004155,0.010656,-0.000351,1.000000,-0.006347,0.008834,0.003066
subscription_type,-0.000881,0.000293,-0.006897,-0.000036,1.000000,-0.002610,-0.002611,0.000850,-0.001205,-0.004539,...,-0.002120,0.002320,-0.002334,-0.000743,0.001023,-0.001362,-0.000036,0.002138,-0.004199,-0.002846
monthly_fee,-0.001249,0.003103,-0.001772,0.002198,-0.002610,1.000000,-0.002279,0.004913,0.005897,-0.005567,...,-0.004023,-0.006194,0.001967,0.005289,0.009575,-0.003728,0.002198,-0.006660,0.004037,-0.004272
payment_method,0.005220,0.003509,-0.006522,0.000750,-0.002611,-0.002279,1.000000,-0.002212,-0.008933,0.006675,...,0.001220,0.001716,0.002169,-0.002216,0.001188,0.004481,0.000750,0.001982,0.002661,0.003013
primary_device,-0.001841,0.000253,0.000674,0.000688,0.000850,0.004913,-0.002212,1.000000,-0.005835,-0.001551,...,0.002061,-0.002166,-0.000913,-0.002303,0.002392,0.001723,0.000688,-0.002470,0.003519,0.005933
devices_used,0.003628,-0.004353,0.002021,-0.002959,-0.001205,0.005897,-0.008933,-0.005835,1.000000,0.001980,...,0.007411,0.002238,0.006901,-0.008248,0.003631,0.001037,-0.002959,0.002118,0.003055,0.007174
favorite_genre,0.005392,-0.009862,-0.009971,0.007346,-0.004539,-0.005567,0.006675,-0.001551,0.001980,1.000000,...,0.002045,0.001441,-0.000621,-0.005220,0.004345,-0.002306,0.007346,0.001440,0.002713,0.003407


In [23]:
print(X.columns)

Index(['age', 'gender', 'country', 'account_age_months', 'subscription_type',
       'monthly_fee', 'payment_method', 'primary_device', 'devices_used',
       'favorite_genre', 'avg_watch_time_minutes', 'watch_sessions_per_week',
       'binge_watch_sessions', 'completion_rate', 'rating_given',
       'content_interactions', 'recommendation_click_rate',
       'days_since_last_login', 'engagement', 'is_active', 'binge_ratio',
       'interaction_score', 'account_age_years', 'prediction', 'low_activity',
       'inactive_user'],
      dtype='str')


In [ ]:
df = df.drop(columns=["prediction"])
X = df.drop("churned", axis=1)
y = df["churned"]
x = pd.get_dummies(X, drop_first=True)


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [26]:
from sklearn.metrics import classification_report

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.80      1.00      0.89      8007
           1       0.00      0.00      0.00      1993

    accuracy                           0.80     10000
   macro avg       0.40      0.50      0.44     10000
weighted avg       0.64      0.80      0.71     10000



c:\Users\geral\miniconda3\envs\netflix_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\geral\miniconda3\envs\netflix_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\geral\miniconda3\envs\netflix_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize